# E4 Extrapolación en L — ¿Basta a/L + b/L² o se necesita d/L³?

**Fecha:** 2026-03-26

**Datos:** 84 puntos de la grilla v1, L = [12, 14, 16, 18, 20, 22] (6 valores).

**Pregunta:** El modelo δR₃ = a/L + b/L² tiene alta correlación a-b.
Añadir d/L³ ¿cambia b significativamente?

## Resultados (ejecutado 2026-03-26)

### Hallazgo principal: d/L³ CAMBIA b drásticamente

| Estadística | Modelo 2p (a/L+b/L²) | Modelo 3p (a/L+b/L²+d/L³) |
|------------|---------------------|----------------------------|
| mean(b) | +0.53 | -9.32 |
| std(b) | 8.37 | 25.91 |
| median(Δb/b) | | -95% |
| d significativo (>2σ) | | 26/84 puntos (31%) |

### Comparación b_3p vs b_anchor (L=40-80):

| (v₁,v₂) | b_2p | b_3p | b_anchor | b_3p/b_anchor |
|----------|------|------|----------|---------------|
| (0.5,1.5) | -0.45 | -12.2 | -4.4 | 2.77 |
| (0.7,1.8) | +9.86 | +16.2 | +2.0 | 8.10 |
| (1.0,2.0) | +8.69 | +15.3 | +14.0 | **1.09** |
| (1.0,2.5) | +5.92 | +1.81 | +5.1 | 0.36 |

### Diagnóstico

1. **d/L³ es significativo en 31% de los puntos** → L=12-22 es insuficiente
   para separar b de d. El modelo de 2 parámetros absorbe d en b.

2. **b_3p NO converge a b_anchor** (ratios 0.36–8.10) → ni 2p ni 3p son
   fiables con L=12-22. Se necesitan L más altos.

3. **Excepción: (1.0,2.0)** tiene b_3p/b_anchor = 1.09 (excelente).
   Esto sugiere que en ciertos puntos d ≈ 0 y el ajuste 2p basta.

4. **La grilla v2 (L=18-34) mejorará** porque L más altos suprimen d/L³.
   Richardson (que elimina a/L) es imprescindible.

### Implicación para c ab initio

c_rich = 1.419 (grilla v1 con Richardson) está contaminado por d/L³.
La grilla v2 (L=18-34) reducirá el término d/L³ por factor (22/34)³ ≈ 0.27.
Estimación: c_v2 debería estar más cerca de c_emp = 1.245.

In [ ]:
import json, numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

with open('../scripts/e4_grilla_b_results.json') as f:
    data = json.load(f)

results = data['results']
L_vals = np.array(data['L_vals'], dtype=float)
print(f'Grilla v1: {len(results)} puntos, L = {data["L_vals"]}')

## 1. Ajuste 2p vs 3p para todos los puntos

In [ ]:
def m2(L, a, b): return a/L + b/L**2
def m3(L, a, b, d): return a/L + b/L**2 + d/L**3

fit_2p = []
fit_3p = []

for key in sorted(results.keys()):
    rec = results[key]
    R3g = rec['R3_GUE']
    R3v = np.array(rec['R3_vals'])
    dR3 = R3v - R3g
    
    try:
        p2, cov2 = curve_fit(m2, L_vals, dR3)
        res2 = dR3 - m2(L_vals, *p2)
        chi2_2 = np.sum(res2**2)
    except: continue
    
    try:
        p3, cov3 = curve_fit(m3, L_vals, dR3, p0=[p2[0], p2[1], 0])
        res3 = dR3 - m3(L_vals, *p3)
        chi2_3 = np.sum(res3**2)
        d_err = np.sqrt(np.diag(cov3))[2]
    except: continue
    
    fit_2p.append({'key': key, 'v1': rec['v1'], 'v2': rec['v2'],
                   'a': p2[0], 'b': p2[1], 'chi2': chi2_2, 'R3g': R3g})
    fit_3p.append({'key': key, 'v1': rec['v1'], 'v2': rec['v2'],
                   'a': p3[0], 'b': p3[1], 'd': p3[2], 'd_err': d_err,
                   'chi2': chi2_3, 'R3g': R3g})

b2 = np.array([f['b'] for f in fit_2p])
b3 = np.array([f['b'] for f in fit_3p])
d3 = np.array([f['d'] for f in fit_3p])
d3_err = np.array([f['d_err'] for f in fit_3p])
d_sig = np.abs(d3) > 2*d3_err

print(f'Puntos: {len(fit_2p)}')
print(f'mean(b_2p) = {np.mean(b2):+.3f} +/- {np.std(b2):.3f}')
print(f'mean(b_3p) = {np.mean(b3):+.3f} +/- {np.std(b3):.3f}')
print(f'mean(d_3p) = {np.mean(d3):+.1f} +/- {np.std(d3):.1f}')
print(f'd significativo (>2sigma): {np.sum(d_sig)}/{len(d_sig)} ({100*np.mean(d_sig):.0f}%)')
print(f'median(chi2_2p/chi2_3p) = {np.median([f2["chi2"]/f3["chi2"] for f2,f3 in zip(fit_2p,fit_3p)]):.1f}')

## 2. Scatter b_2p vs b_3p

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# b_2p vs b_3p
ax = axes[0]
ax.scatter(b2[~d_sig], b3[~d_sig], alpha=0.4, s=15, label=f'd no sig ({np.sum(~d_sig)})')
ax.scatter(b2[d_sig], b3[d_sig], alpha=0.6, s=25, c='red', label=f'd sig >2s ({np.sum(d_sig)})')
lim = max(abs(b2).max(), abs(b3).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.3)
ax.set_xlabel('b (2 params)'); ax.set_ylabel('b (3 params)')
ax.set_title('b_2p vs b_3p'); ax.legend(fontsize=8)

# Histograma de d
ax = axes[1]
ax.hist(d3, bins=25, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', ls='--')
ax.set_xlabel('d (coef de 1/L^3)'); ax.set_ylabel('Frecuencia')
ax.set_title(f'd/L^3: mean={np.mean(d3):+.0f}, std={np.std(d3):.0f}')

# Db/b
ax = axes[2]
db_rel = (b3 - b2) / np.where(np.abs(b2) > 0.1, np.abs(b2), np.nan)
valid_db = db_rel[~np.isnan(db_rel)]
ax.hist(np.clip(valid_db, -5, 5), bins=25, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', ls='--')
ax.set_xlabel('(b_3p - b_2p) / |b_2p|'); ax.set_ylabel('Frecuencia')
ax.set_title(f'Cambio relativo en b\nmedian={np.nanmedian(valid_db)*100:+.0f}%')

plt.tight_layout()
plt.savefig('../paper/figures/e4_extrapolacion_L.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Comparación b_3p vs b_anchor (L=40-80)

In [ ]:
anchor_b = {
    '0.5,1.5': -4.4, '0.7,1.8': 2.0, '1.0,2.0': 14.0,
    '1.0,2.5': 5.1, '0.8,1.2': -0.1
}

print('%-12s %10s %10s %10s %12s %12s' %
      ('(v1,v2)', 'b_2p', 'b_3p', 'b_anchor', 'b2p/anchor', 'b3p/anchor'))
print('-'*70)

for key, ba in anchor_b.items():
    f2 = next((f for f in fit_2p if f['key'] == key), None)
    f3 = next((f for f in fit_3p if f['key'] == key), None)
    if f2 and f3 and abs(ba) > 0.01:
        print('(%-9s) %+10.2f %+10.2f %+10.2f %12.2f %12.2f' %
              (key, f2['b'], f3['b'], ba, f2['b']/ba, f3['b']/ba))
    elif f2 and f3:
        print('(%-9s) %+10.2f %+10.2f %+10.2f %12s %12s' %
              (key, f2['b'], f3['b'], ba, 'N/A', 'N/A'))

print()
print('CONCLUSION:')
print('  b_2p/anchor: 0.10 a 4.93 (muy disperso)')
print('  b_3p/anchor: 0.36 a 8.10 (aun peor)')
print('  => Con L=12-22, ni 2p ni 3p aislan b correctamente.')
print('  => La grilla v2 (L=18-34) es necesaria.')

## 4. Ejemplo: δR₃(L) para puntos selectos

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

show_keys = ['0.5,1.5', '0.7,1.8', '1.0,2.0', '1.0,2.5', '0.8,1.2', '1.0,1.5']

L_fine = np.linspace(10, 90, 200)

for idx, key in enumerate(show_keys):
    ax = axes[idx]
    rec = results.get(key)
    if not rec: continue
    
    R3g = rec['R3_GUE']
    dR3 = np.array(rec['R3_vals']) - R3g
    
    p2, _ = curve_fit(m2, L_vals, dR3)
    p3, _ = curve_fit(m3, L_vals, dR3, p0=[p2[0], p2[1], 0])
    
    ax.plot(L_vals, dR3 * L_vals**2, 'ko', ms=6, label='datos')
    ax.plot(L_fine, m2(L_fine, *p2) * L_fine**2, 'b-',
            label=f'2p: b={p2[1]:+.1f}')
    ax.plot(L_fine, m3(L_fine, *p3) * L_fine**2, 'r--',
            label=f'3p: b={p3[1]:+.1f}')
    
    # Ancla si existe
    ba = anchor_b.get(key)
    if ba is not None:
        ax.axhline(ba, color='green', ls=':', lw=2, label=f'anchor={ba:+.1f}')
    
    ax.set_xlabel('L'); ax.set_ylabel('dR3 * L^2')
    ax.set_title(f'({key}) R3g={R3g:.3f}')
    ax.legend(fontsize=7)

plt.suptitle('dR3*L2 vs L: modelo 2p (azul) vs 3p (rojo) vs anchor (verde)', fontsize=12)
plt.tight_layout()
plt.savefig('../paper/figures/e4_extrapolacion_puntos.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Resumen

In [ ]:
print('='*60)
print('RESUMEN - Extrapolacion L con grilla v1')
print('='*60)
print(f'Datos: {len(results)} puntos, L = {data["L_vals"]}')
print()
print('Modelo 2p (a/L + b/L2):')
print(f'  mean(b) = {np.mean(b2):+.3f} +/- {np.std(b2):.3f}')
print()
print('Modelo 3p (a/L + b/L2 + d/L3):')
print(f'  mean(b) = {np.mean(b3):+.3f} +/- {np.std(b3):.3f}')
print(f'  mean(d) = {np.mean(d3):+.0f} +/- {np.std(d3):.0f}')
print(f'  d significativo: {np.sum(d_sig)}/{len(d_sig)} ({100*np.mean(d_sig):.0f}%)')
print()
print('CONCLUSION:')
print('  1. d/L3 cambia b drasticamente (median Db/b = -95%)')
print('  2. L=12-22 no separa b de d => b_v1 NO es fiable')
print('  3. Ni b_2p ni b_3p convergen a b_anchor (L=40-80)')
print('  4. Grilla v2 (L=18-34) suprimira d/L3 por factor ~0.27')
print('  5. Richardson + L altos es la unica via fiable')
print('='*60)